In [2]:
%pip install kaggle


     -------------------------------------- 181.2/181.2 kB 1.1 MB/s eta 0:00:00
     -------------------------------------- 165.1/165.1 kB 3.3 MB/s eta 0:00:00
     -------------------------------------- 135.7/135.7 kB 3.9 MB/s eta 0:00:00
     -------------------------------------- 135.1/135.1 kB 2.7 MB/s eta 0:00:00
     ---------------------------------------- 69.6/69.6 kB 1.9 MB/s eta 0:00:00
     -------------------------------------- 215.4/215.4 kB 4.4 MB/s eta 0:00:00
     ---------------------------------------- 73.1/73.1 kB 3.9 MB/s eta 0:00:00
     ---------------------------------------- 78.2/78.2 kB 2.2 MB/s eta 0:00:00
     ---------------------------------------- 80.2/80.2 kB 1.5 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import polars as pl

col = "event_original_message"
files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.parquet")))

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in files:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(3).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: event_original_message
Total rows: 25814017
Non-null values: 25813845
Missing values: 172
Non-null percentage: 99.999334 %
Files containing values: 130

Sample values:
1. 'Id = {00000000-0000-0000-0000-000000000000}; ClientMachine = DESKTOP-4PVPS6E; User = NT AUTHORITY\\SYSTEM; ClientProcessId = 4264; Component = Unknown; Operation = Start IWbemServices::ExecQuery - root\\CIMV2 : SELECT uuid FROM win32_computersystemprod'
2. 'Id = {00000000-0000-0000-0000-000000000000}; ClientMachine = DESKTOP-4PVPS6E; User = NT AUTHORITY\\SYSTEM; ClientProcessId = 4264; Component = Unknown; Operation = Start IWbemServices::ExecQuery - root\\Microsoft\\Windows\\DeviceGuard : SELECT SecuritySe'
3. 'Id = {00000000-0000-0000-0000-000000000000}; ClientMachine = DESKTOP-4PVPS6E; User = NT AUTHORITY\\SYSTEM; ClientProcessId = 2248; Component = Unknown; Operation = Start IWbemServices::ExecQuery - root\\wmi : select * from WDMClassesOfDriver where Driv'
4. 'Auditing settings on object were changed.\n\

In [12]:


import os
import glob
import re
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# =====================================================
# 1. Paths and settings
# =====================================================

INPUT_DIR = r"C:\Users\Administrator\Documents\comiset\data\comiset_parquet_v12"
OUTPUT_DIR = r"C:\Users\Administrator\Documents\comiset\data\comiset_parquet_v12_FE3"

BATCH_SIZE = 50_000

files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.parquet")))

if len(files) != 130:
    raise ValueError(f"Expected 130 files, found {len(files)}")

if os.path.exists(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    raise FileExistsError(
        f"Output folder is not empty: {OUTPUT_DIR}\n"
        "Use a new output folder or inspect existing results."
    )

os.makedirs(OUTPUT_DIR, exist_ok=True)

FE_COLUMNS = [
    "CommandLine", "ImagePath", "ImageLoaded", "Operation", "Binary",
    "CallTrace", "z_elastic_ecs", "Details", "TargetFilename",
    "TargetObject", "event_original_message", "object_name", "HiveName",
    "Message", "process_path", "target_process_path", "ActionName",
    "QueryResults", "process_parent_path", "src_ip_addr"
]

input_schema = pq.read_schema(files[0])
missing = [c for c in FE_COLUMNS if c not in input_schema.names]

if missing:
    raise ValueError(f"Missing FE columns: {missing}")

KEEP_COLUMNS = [
    c for c in input_schema.names if c not in FE_COLUMNS
]

READ_COLUMNS = input_schema.names

# =====================================================
# 2. Feature engineering
# =====================================================

def generate_features(df):
    features = {}

    def s(name):
        return df[name].astype("string").fillna("").str.lower()

    def flag(name, feature_name, pattern, regex=True):
        features[feature_name] = (
            s(name).str.contains(pattern, regex=regex, na=False)
            .astype("int8")
        )

    # 1. process_path
    x = s("process_path")
    features["process_path_present"] = x.str.strip().ne("").astype("int8")
    flag("process_path", "process_is_system32", r"\\system32\\")
    flag("process_path", "process_is_program_files", r"\\program files(?: \(x86\))?\\")
    flag("process_path", "process_is_temp", r"\\temp\\|\\appdata\\local\\temp\\")
    flag("process_path", "process_is_windows_defender", r"\\windows defender\\|\\msmpeng\.exe")
    features["process_path_length"] = x.str.len().fillna(0).astype("int32")
    features["process_path_depth"] = x.str.count(r"\\").fillna(0).astype("int16")
    features["process_extension_is_exe"] = x.str.endswith(".exe").astype("int8")

    # 2. target_process_path
    x = s("target_process_path")
    features["target_process_present"] = x.str.strip().ne("").astype("int8")
    features["target_is_registry"] = x.str.strip().eq("registry").astype("int8")
    flag("target_process_path", "target_is_system32", r"\\system32\\")
    flag("target_process_path", "target_is_temp", r"\\temp\\|\\appdata\\local\\temp\\")
    flag("target_process_path", "target_is_program_files", r"\\program files(?: \(x86\))?\\")
    features["target_process_path_length"] = x.str.len().fillna(0).astype("int32")
    features["target_process_extension_is_exe"] = x.str.endswith(".exe").astype("int8")

    # 3. process_parent_path
    x = s("process_parent_path")
    features["parent_process_present"] = x.str.strip().ne("").astype("int8")
    flag("process_parent_path", "parent_is_system32", r"\\system32\\")
    flag("process_parent_path", "parent_is_temp", r"\\temp\\|\\appdata\\local\\temp\\")
    flag("process_parent_path", "parent_is_program_files", r"\\program files(?: \(x86\))?\\")
    features["parent_process_extension_is_exe"] = x.str.endswith(".exe").astype("int8")

    # 4. ImagePath
    x = s("ImagePath")
    features["imagepath_present"] = x.str.strip().ne("").astype("int8")
    features["imagepath_is_driver"] = x.str.endswith(".sys").astype("int8")
    features["imagepath_is_exe"] = x.str.endswith(".exe").astype("int8")
    flag("ImagePath", "imagepath_is_system32", r"\\system32\\")
    flag("ImagePath", "imagepath_is_defender", r"defender|msmpeng")

    # 5. object_name
    x = s("object_name")
    features["object_name_present"] = x.str.strip().ne("").astype("int8")
    flag("object_name", "object_is_temp", r"\\temp\\|\\appdata\\local\\temp\\")
    flag("object_name", "object_is_winsxs", r"\\winsxs\\")
    flag("object_name", "object_is_system32", r"\\system32\\")
    features["object_name_length"] = x.str.len().fillna(0).astype("int32")

    # 6. HiveName
    x = s("HiveName")
    features["hivename_present"] = x.str.strip().ne("").astype("int8")
    flag("HiveName", "hivename_is_system_config", r"system|config")
    flag("HiveName", "hivename_is_apprepository", "apprepository", regex=False)
    flag("HiveName", "hivename_is_user_profile", r"users|user profile|hku")
    features["hivename_length"] = x.str.len().fillna(0).astype("int32")

    # 7. TargetObject
    x = s("TargetObject")
    features["targetobject_present"] = x.str.strip().ne("").astype("int8")
    for term, feature_name in [
        ("hklm", "targetobject_is_hklm"),
        ("hku", "targetobject_is_hku"),
        ("tcpip", "targetobject_has_tcpip"),
        ("certificate", "targetobject_has_certificate"),
        ("services", "targetobject_has_services"),
        ("software", "targetobject_has_software"),
        ("appcontainer", "targetobject_has_appcontainer"),
    ]:
        features[feature_name] = x.str.contains(term, regex=False).astype("int8")
    features["targetobject_length"] = x.str.len().fillna(0).astype("int32")
    features["targetobject_depth"] = x.str.count(r"\\").fillna(0).astype("int16")

    # 8. Operation
    x = s("Operation")
    features["operation_present"] = x.str.strip().ne("").astype("int8")
    flag("Operation", "operation_is_service_start", r"start.*service|service.*start")
    flag("Operation", "operation_is_service_stop", r"stop.*service|service.*stop")
    flag("Operation", "operation_is_wmi", "wmi", regex=False)
    features["operation_length"] = x.str.len().fillna(0).astype("int32")

    # 9. Details
    x = s("Details")
    features["details_present"] = x.str.strip().ne("").astype("int8")
    flag("Details", "details_is_binary_data", "binary data", regex=False)
    flag("Details", "details_is_dword", "dword", regex=False)
    flag("Details", "details_is_qword", "qword", regex=False)
    flag("Details", "details_has_hex_value", r"0x[0-9a-f]+")
    features["details_length"] = x.str.len().fillna(0).astype("int32")

    # 10. ActionName
    x = s("ActionName")
    features["actionname_present"] = x.str.strip().ne("").astype("int8")
    flag("ActionName", "actionname_is_executable_path", r"\.exe(?:$|\s)")
    flag("ActionName", "actionname_is_system32", r"\\system32\\")
    flag("ActionName", "actionname_is_task_handler", "taskhandler", regex=False)
    features["actionname_length"] = x.str.len().fillna(0).astype("int32")

    # 11. CommandLine
    x = s("CommandLine")
    features["cmd_present"] = x.str.strip().ne("").astype("int8")
    features["cmd_length"] = x.str.len().fillna(0).astype("int32")
    features["cmd_argument_count"] = x.str.split().str.len().fillna(0).astype("int16")
    flag("CommandLine", "cmd_has_encoded_flag", r"-encodedcommand|-enc\b")
    flag("CommandLine", "cmd_has_powershell", "powershell", regex=False)
    flag("CommandLine", "cmd_has_temp_path", r"\\temp\\|\\appdata\\local\\temp\\")
    flag("CommandLine", "cmd_has_admin_share", r"\\\\[^\\]+\\(?:admin\$|c\$)")

    # 12. ImageLoaded
    x = s("ImageLoaded")
    features["imageloaded_present"] = x.str.strip().ne("").astype("int8")
    features["imageloaded_is_dll"] = x.str.endswith(".dll").astype("int8")
    features["imageloaded_is_sys"] = x.str.endswith(".sys").astype("int8")
    features["imageloaded_is_exe"] = x.str.endswith(".exe").astype("int8")
    flag("ImageLoaded", "imageloaded_is_system32", r"\\system32\\")
    flag("ImageLoaded", "imageloaded_is_temp", r"\\temp\\|\\appdata\\local\\temp\\")
    features["imageloaded_length"] = x.str.len().fillna(0).astype("int32")

    # 13. TargetFilename
    x = s("TargetFilename")
    features["targetfilename_present"] = x.str.strip().ne("").astype("int8")
    features["targetfilename_length"] = x.str.len().fillna(0).astype("int32")
    flag("TargetFilename", "targetfilename_is_prefetch", r"\\prefetch\\")
    flag("TargetFilename", "targetfilename_is_temp", r"\\temp\\|\\appdata\\local\\temp\\")
    flag("TargetFilename", "targetfilename_is_system32", r"\\system32\\")
    features["targetfilename_is_log_or_etl"] = x.str.endswith((".log", ".etl")).astype("int8")
    features["targetfilename_is_executable"] = x.str.endswith((".exe", ".dll", ".sys")).astype("int8")

    # 14. CallTrace
    x = s("CallTrace")
    features["calltrace_present"] = x.str.strip().ne("").astype("int8")
    features["calltrace_length"] = x.str.len().fillna(0).astype("int32")
    flag("CallTrace", "calltrace_has_ntdll", "ntdll", regex=False)
    flag("CallTrace", "calltrace_has_kernelbase", "kernelbase", regex=False)
    flag("CallTrace", "calltrace_has_defender_module", r"mpengine|wd\w+\.dll|defender")

    # Binary: no generated feature; original Binary column is dropped.

    # 15. event_original_message
    x = s("event_original_message")
    features["original_message_present"] = x.str.strip().ne("").astype("int8")
    features["original_message_length"] = x.str.len().fillna(0).astype("int32")
    features["original_message_line_count"] = (
        x.str.count(r"\n").fillna(0) + x.str.strip().ne("").astype("int8")
    ).astype("int16")

    # 16. Message
    x = s("Message")
    features["message_present"] = x.str.strip().ne("").astype("int8")
    features["message_length"] = x.str.len().fillna(0).astype("int32")

    # 17. QueryResults
    x = s("QueryResults")
    features["queryresults_present"] = x.str.strip().ne("").astype("int8")
    features["queryresults_length"] = x.str.len().fillna(0).astype("int32")
    features["queryresults_ip_count"] = x.str.count(
        r"\b(?:\d{1,3}\.){3}\d{1,3}\b|(?:[0-9a-f]{0,4}:){2,}[0-9a-f:]*"
    ).fillna(0).astype("int16")

    # 18. src_ip_addr
    x = s("src_ip_addr")
    features["src_ip_present"] = x.str.strip().ne("").astype("int8")
    features["src_ip_is_private"] = x.str.match(
        r"^(10\.|192\.168\.|172\.(?:1[6-9]|2\d|3[01])\.)",
        na=False
    ).astype("int8")
    features["src_ip_is_loopback"] = x.isin(["127.0.0.1", "::1"]).astype("int8")
    features["src_ip_is_ipv6"] = x.str.contains(":", regex=False).astype("int8")

    # 19. z_elastic_ecs
    x = s("z_elastic_ecs")
    features["ecs_present"] = x.str.strip().ne("").astype("int8")
    features["ecs_length"] = x.str.len().fillna(0).astype("int32")
    flag("z_elastic_ecs", "ecs_has_event_code", '"code"', regex=False)
    flag("z_elastic_ecs", "ecs_has_provider", '"provider"', regex=False)
    flag("z_elastic_ecs", "ecs_has_outcome", '"outcome"', regex=False)

    return pd.DataFrame(features, index=df.index)


# =====================================================
# 3. Process files and batches
# =====================================================

feature_count = None
total_rows = 0
completed = 0

for file_index, input_path in enumerate(files, start=1):
    filename = os.path.basename(input_path)
    output_path = os.path.join(OUTPUT_DIR, filename)

    print(f"\n[{file_index}/130] Processing {filename}", flush=True)

    parquet_file = pq.ParquetFile(input_path)
    input_rows = parquet_file.metadata.num_rows
    writer = None
    file_rows = 0

    try:
        for batch in parquet_file.iter_batches(
            batch_size=BATCH_SIZE,
            columns=READ_COLUMNS
        ):
            # Convert selected columns for FE
            fe_table = batch.select(FE_COLUMNS)
            df = fe_table.to_pandas()

            feature_df = generate_features(df)

            # Retain original columns except the 20 FE source columns
            output_table = batch.select(KEEP_COLUMNS)

            # Build all generated features together, then append once
            feature_table = pa.Table.from_pandas(
                feature_df, preserve_index=False
            )

            output_table = pa.Table.from_arrays(
                list(output_table.columns) + list(feature_table.columns),
                names=output_table.column_names + feature_table.column_names
            )

            if writer is None:
                writer = pq.ParquetWriter(
                    output_path,
                    output_table.schema,
                    compression="zstd"
                )
                if feature_count is None:
                    feature_count = len(feature_df.columns)

            writer.write_table(output_table, row_group_size=BATCH_SIZE)

            batch_rows = output_table.num_rows
            file_rows += batch_rows
            total_rows += batch_rows

            del batch, fe_table, df, feature_df, feature_table, output_table

        if writer is not None:
            writer.close()
            writer = None

        gc.collect()

        # Verify the saved file
        output_pf = pq.ParquetFile(output_path)
        output_rows = output_pf.metadata.num_rows
        output_cols = len(output_pf.schema_arrow.names)

        expected_cols = len(KEEP_COLUMNS) + feature_count

        if output_rows != input_rows:
            raise RuntimeError(
                f"Row mismatch in {filename}: input={input_rows}, output={output_rows}"
            )

        if output_cols != expected_cols:
            raise RuntimeError(
                f"Column mismatch in {filename}: expected={expected_cols}, got={output_cols}"
            )

        completed += 1
        print(
            f"Verified: {output_rows:,} rows | {output_cols:,} columns "
            f"| {feature_count} new features",
            flush=True
        )

    except Exception:
        if writer is not None:
            writer.close()
        print(f"ERROR while processing {filename}. Stop and inspect output.", flush=True)
        raise

print("\n======================================")
print("FEATURE ENGINEERING FINISHED")
print("Files completed:", completed, "/", len(files))
print("Total output rows:", f"{total_rows:,}")
print("Generated features per row:", feature_count)
print("Output folder:", OUTPUT_DIR)
print("Original input files were not modified.")
print("======================================")



[1/130] Processing part-00000.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[2/130] Processing part-00001.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[3/130] Processing part-00002.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[4/130] Processing part-00003.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[5/130] Processing part-00004.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[6/130] Processing part-00005.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[7/130] Processing part-00006.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[8/130] Processing part-00007.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[9/130] Processing part-00008.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[10/130] Processing part-00009.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[11/130] Processin

In [22]:

import os
import glob
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

INPUT_DIR = r"C:\Users\Administrator\Documents\comiset\data\comiset_parquet_v12"
OUTPUT_DIR = r"C:\Users\Administrator\Documents\comiset\data\comiset_parquet_v12_FE3"

input_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.parquet")))
output_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.parquet")))

assert len(input_files) == 130, f"Input files: {len(input_files)}"
assert len(output_files) == 130, f"Output files: {len(output_files)}"


original_columns = pq.read_schema(input_files[0]).names
output_columns = pq.read_schema(output_files[0]).names
features = [c for c in output_columns if c not in original_columns]


count_suffixes = (
    "_length", "_depth", "_count",
    "_line_count", "_ip_count", "_argument_count"
)
count_features = [
    c for c in features if c.endswith(count_suffixes)
]


binary_features = [
    c for c in features if c not in count_features
]


indices = sorted(set([
    0, len(output_files)//4, len(output_files)//2,
    (3*len(output_files))//4, len(output_files)-1
]))
sample_files = [output_files[i] for i in indices]


frames = []

for path in sample_files:
    parquet_file = pq.ParquetFile(path)
    batches = parquet_file.iter_batches(
        batch_size=10_000,
        columns=features
    )
    batch = next(batches, None)

    if batch is not None:
        frames.append(batch.to_pandas())

df = pd.concat(frames, ignore_index=True)

print("Generated features:", len(features))
print("Sample rows:", f"{len(df):,}")
print("Binary features:", len(binary_features))
print("Length/count features:", len(count_features))


missing_cells = int(df[features].isna().sum().sum())
print("\nMissing/NaN cells:", missing_cells)


infinite_cells = 0
for col in features:
    values = pd.to_numeric(df[col], errors="coerce").to_numpy(
        dtype=float, na_value=np.nan
    )
    infinite_cells += int(np.isinf(values).sum())

print("Infinite values:", infinite_cells)


binary_issues = {}

for col in binary_features:
    values = set(df[col].dropna().unique().tolist())
    if not values.issubset({0, 1, False, True}):
        binary_issues[col] = sorted(map(str, values))[:10]

print("Binary features with invalid values:", len(binary_issues))
if binary_issues:
    print("Examples:", dict(list(binary_issues.items())[:10]))

negative_issues = {}

for col in count_features:
    values = pd.to_numeric(df[col], errors="coerce").dropna()
    negatives = int((values < 0).sum())

    if negatives:
        negative_issues[col] = negatives

print("Length/count features with negative values:",
      len(negative_issues))
if negative_issues:
    print("Examples:", negative_issues)


print("\nSample feature statistics:")
show_columns = (binary_features[:5] + count_features[:10])
show_columns = list(dict.fromkeys(show_columns))

if show_columns:
    print(df[show_columns].describe().to_string())


if (missing_cells == 0
        and infinite_cells == 0
        and not binary_issues
        and not negative_issues):
    print("\nPASS: All sampled value checks passed.")
else:
    print("\nREVIEW: Some sampled values need investigation.")


Generated features: 104
Sample rows: 50,000
Binary features: 83
Length/count features: 21

Missing/NaN cells: 0
Infinite values: 0
Binary features with invalid values: 0
Length/count features with negative values: 0

Sample feature statistics:
       process_path_present  process_is_system32  process_is_program_files  process_is_temp  process_is_windows_defender  process_path_length  process_path_depth  target_process_path_length  object_name_length  hivename_length  targetobject_length  targetobject_depth  operation_length  details_length  actionname_length
count          50000.000000         50000.000000              50000.000000          50000.0                 50000.000000         50000.000000         50000.00000                50000.000000             50000.0     50000.000000         50000.000000        50000.000000      50000.000000    50000.000000       50000.000000
mean               0.600960             0.219240                  0.249580              0.0                     0.

In [25]:
import os
import glob
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

INPUT_DIR = r"C:\Users\Administrator\Documents\comiset\data\comiset_parquet_v12"
OUTPUT_DIR = r"C:\Users\Administrator\Documents\comiset\data\comiset_parquet_v12_FE3"
REPORT_DIR = os.path.join(OUTPUT_DIR, "feature_review_reports")
os.makedirs(REPORT_DIR, exist_ok=True)

input_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.parquet")))
output_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.parquet")))
if len(input_files) != 130 or len(output_files) != 130:
    raise ValueError(f"Expected 130 input/output files; got {len(input_files)}/{len(output_files)}")

original_cols = pq.read_schema(input_files[0]).names
out_schema = pq.read_schema(output_files[0])
features = [c for c in out_schema.names if c not in original_cols]

# Read a small sample spread across files; only generated feature columns.
sample_indices = sorted(set([0, len(output_files)//4, len(output_files)//2,
                             (3*len(output_files))//4, len(output_files)-1]))
frames = []
for idx in sample_indices:
    pf = pq.ParquetFile(output_files[idx])
    batch = next(pf.iter_batches(batch_size=10_000, columns=features), None)
    if batch is not None:
        frames.append(batch.to_pandas())

df = pd.concat(frames, ignore_index=True)
n = len(df)
rows = []

for col in features:
    s = df[col]
    nonnull = s.dropna()
    nunique = int(nonnull.nunique())
    missing = int(s.isna().sum())
    dtype = str(s.dtype)
    if nunique <= 1:
        status = "CONSTANT_IN_SAMPLE"
    elif nunique == 2 and set(nonnull.unique()).issubset({0, 1, False, True}):
        status = "BINARY"
    else:
        status = "NUMERIC_VARIATION"
    rows.append({
        "feature": col,
        "dtype": dtype,
        "sample_rows": n,
        "missing_count": missing,
        "unique_nonnull_values": nunique,
        "zero_count": int((s == 0).sum()) if pd.api.types.is_numeric_dtype(s) or str(s.dtype) == "bool" else None,
        "zero_pct": round(100 * float((s == 0).sum()) / max(n, 1), 3) if pd.api.types.is_numeric_dtype(s) or str(s.dtype) == "bool" else None,
        "min": nonnull.min() if len(nonnull) else None,
        "max": nonnull.max() if len(nonnull) else None,
        "status": status
    })

profile = pd.DataFrame(rows)
constant = profile[profile["status"] == "CONSTANT_IN_SAMPLE"]
print("Rows sampled:", f"{n:,}")
print("Features reviewed:", len(features))
print("Features constant in this sample:", len(constant))
print("Features with missing values in sample:", int((profile["missing_count"] > 0).sum()))
print("Features with >99.9% zeros:", int((profile["zero_pct"] > 99.9).sum()))
print("\nConstant-in-sample features:")
print(constant[["feature", "unique_nonnull_values", "zero_pct", "min", "max"]].to_string(index=False)
      if len(constant) else "None")

sparse = profile[profile["zero_pct"] > 99.9].sort_values("zero_pct", ascending=False)
print("\nFeatures with >99.9% zeros (sample only):")
print(sparse[["feature", "zero_pct", "unique_nonnull_values", "min", "max"]].to_string(index=False)
      if len(sparse) else "None")

# Exact duplicates within the sampled rows; compare numeric vectors.
duplicate_groups = []
seen = {}
for col in features:
    # Include NaN consistently in signature by using pandas hashing.
    sig = pd.util.hash_pandas_object(df[col], index=False).values.tobytes()
    if sig in seen:
        duplicate_groups.append((seen[sig], col))
    else:
        seen[sig] = col

print("\nExact matching feature pairs in sample:", len(duplicate_groups))
for a, b in duplicate_groups[:40]:
    print(f"  {a} == {b}")

profile_path = os.path.join(REPORT_DIR, "feature_profile_sample.csv")
duplicates_path = os.path.join(REPORT_DIR, "feature_duplicate_pairs_sample.csv")
profile.to_csv(profile_path, index=False)
pd.DataFrame(duplicate_groups, columns=["feature_1", "feature_2"]).to_csv(duplicates_path, index=False)

print("\nSaved reports:")
print(profile_path)
print(duplicates_path)
print("\nIMPORTANT: These are sample-based screening results, not proof that a feature is globally constant or redundant.")


Rows sampled: 50,000
Features reviewed: 104
Features constant in this sample: 20
Features with missing values in sample: 0
Features with >99.9% zeros: 34

Constant-in-sample features:
                feature  unique_nonnull_values  zero_pct  min  max
        process_is_temp                      1     100.0    0    0
     target_is_registry                      1     100.0    0    0
         target_is_temp                      1     100.0    0    0
         parent_is_temp                      1     100.0    0    0
parent_is_program_files                      1     100.0    0    0
    object_name_present                      1     100.0    0    0
         object_is_temp                      1     100.0    0    0
       object_is_winsxs                      1     100.0    0    0
     object_is_system32                      1     100.0    0    0
     object_name_length                      1     100.0    0    0
   cmd_has_encoded_flag                      1     100.0    0    0
     cmd_has

In [26]:

import os
import glob
import pandas as pd
import pyarrow.parquet as pq

OUTPUT_DIR = r"C:\Users\Administrator\Documents\comiset\data\comiset_parquet_v12_FE3"
REPORT_DIR = os.path.join(OUTPUT_DIR, "feature_review_reports")

os.makedirs(REPORT_DIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.parquet")))

if len(files) != 130:
    raise ValueError(f"Expected 130 files, found {len(files)}")

# Features that were constant in the 50,000-row sample
suspect_features = [
    "process_is_temp",
    "target_is_registry",
    "target_is_temp",
    "parent_is_temp",
    "parent_is_program_files",
    "object_name_present",
    "object_is_temp",
    "object_is_winsxs",
    "object_is_system32",
    "object_name_length",
    "cmd_has_encoded_flag",
    "cmd_has_powershell",
    "cmd_has_temp_path",
    "cmd_has_admin_share",
    "imageloaded_is_temp",
    "src_ip_is_loopback",
    "ecs_present",
    "ecs_has_event_code",
    "ecs_has_provider",
    "ecs_has_outcome",
]

# Pairs found identical in the sample
pairs = [
    ("process_is_temp", "target_is_registry"),
    ("process_is_temp", "target_is_temp"),
    ("process_is_temp", "parent_is_temp"),
    ("process_is_temp", "parent_is_program_files"),
    ("parent_process_present", "parent_process_extension_is_exe"),
    ("process_is_temp", "object_name_present"),
    ("process_is_temp", "object_is_temp"),
    ("process_is_temp", "object_is_winsxs"),
    ("process_is_temp", "object_is_system32"),
    ("process_is_temp", "object_name_length"),
    ("parent_process_present", "cmd_present"),
    ("process_is_temp", "cmd_has_encoded_flag"),
    ("process_is_temp", "cmd_has_powershell"),
    ("process_is_temp", "cmd_has_temp_path"),
    ("process_is_temp", "cmd_has_admin_share"),
    ("process_is_temp", "imageloaded_is_temp"),
    ("calltrace_present", "calltrace_has_ntdll"),
    ("process_is_temp", "src_ip_is_loopback"),
    ("ecs_present", "ecs_has_event_code"),
    ("ecs_present", "ecs_has_provider"),
    ("process_is_temp", "ecs_has_outcome"),
]

schema_cols = pq.read_schema(files[0]).names
all_suspects = set(suspect_features)
for a, b in pairs:
    all_suspects.add(a)
    all_suspects.add(b)

missing_cols = sorted(all_suspects - set(schema_cols))
if missing_cols:
    raise ValueError(f"Columns not found in output schema: {missing_cols}")

columns = sorted(all_suspects)

# Track whether each feature ever has a nonzero value,
# its min/max, and the number of missing values.
stats = {
    c: {"rows": 0, "missing": 0, "nonzero": 0,
        "min": None, "max": None, "unique_values": set()}
    for c in columns
}

pair_mismatches = {(a, b): 0 for a, b in pairs}
total_rows = 0
BATCH_SIZE = 50_000

for file_no, path in enumerate(files, start=1):
    pf = pq.ParquetFile(path)

    for batch in pf.iter_batches(
        batch_size=BATCH_SIZE,
        columns=columns
    ):
        df = batch.to_pandas()
        total_rows += len(df)

        for c in columns:
            s = df[c]
            st = stats[c]
            st["rows"] += len(s)
            st["missing"] += int(s.isna().sum())

            numeric = pd.to_numeric(s, errors="coerce")
            st["nonzero"] += int((numeric.fillna(0) != 0).sum())

            if numeric.notna().any():
                lo = numeric.min()
                hi = numeric.max()
                st["min"] = lo if st["min"] is None else min(st["min"], lo)
                st["max"] = hi if st["max"] is None else max(st["max"], hi)

            st["unique_values"].update(numeric.dropna().unique().tolist())

        for a, b in pairs:
            # Nulls count as equal only when both are null.
            left = df[a]
            right = df[b]
            equal = left.eq(right) | (left.isna() & right.isna())
            pair_mismatches[(a, b)] += int((~equal).sum())

    if file_no % 10 == 0 or file_no == len(files):
        print(f"Processed {file_no}/{len(files)} files")

# Summary of suspected constants
constant_rows = []
for c in suspect_features:
    st = stats[c]
    constant_rows.append({
        "feature": c,
        "rows_checked": st["rows"],
        "missing": st["missing"],
        "nonzero_values": st["nonzero"],
        "unique_nonnull_values": len(st["unique_values"]),
        "min": st["min"],
        "max": st["max"],
        "globally_constant_in_checked_data":
            len(st["unique_values"]) <= 1
    })

constant_report = pd.DataFrame(constant_rows)
constant_report.to_csv(
    os.path.join(REPORT_DIR, "full_scan_suspect_features.csv"),
    index=False
)

pair_report = pd.DataFrame([
    {
        "feature_1": a,
        "feature_2": b,
        "rows_checked": total_rows,
        "mismatches": mismatch_count,
        "identical_across_all_rows": mismatch_count == 0
    }
    for (a, b), mismatch_count in pair_mismatches.items()
])
pair_report.to_csv(
    os.path.join(REPORT_DIR, "full_scan_duplicate_pairs.csv"),
    index=False
)

print("\nTotal rows scanned:", f"{total_rows:,}")

print("\nSuspected constant features:")
print(constant_report.to_string(index=False))

print("\nPairs still identical across all rows:")
print(pair_report[pair_report["identical_across_all_rows"]].to_string(index=False))

print("\nPairs that differ somewhere:")
print(pair_report[~pair_report["identical_across_all_rows"]].to_string(index=False))

print("\nReports saved in:", REPORT_DIR)


Processed 10/130 files
Processed 20/130 files
Processed 30/130 files
Processed 40/130 files
Processed 50/130 files
Processed 60/130 files
Processed 70/130 files
Processed 80/130 files
Processed 90/130 files
Processed 100/130 files
Processed 110/130 files
Processed 120/130 files
Processed 130/130 files

Total rows scanned: 25,814,017

Suspected constant features:
                feature  rows_checked  missing  nonzero_values  unique_nonnull_values  min  max  globally_constant_in_checked_data
        process_is_temp      25814017        0            2402                      2    0    1                              False
     target_is_registry      25814017        0             838                      2    0    1                              False
         target_is_temp      25814017        0             705                      2    0    1                              False
         parent_is_temp      25814017        0              22                      2    0    1                

In [32]:

import os
import glob
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

SOURCE_DIR = r"C:\Users\Administrator\Documents\comiset\data\comiset_parquet_v12_FE3"
FINAL_DIR = r"C:\Users\Administrator\Documents\comiset\data\comiset_parquet_v12_FINAL"

REMOVE = [
    "ecs_present",
    "ecs_has_event_code",
    "ecs_has_provider",
    "calltrace_has_ntdll",
]

# 1. Check source and destination
if not os.path.isdir(SOURCE_DIR):
    raise FileNotFoundError(f"Source folder not found: {SOURCE_DIR}")

if os.path.exists(FINAL_DIR):
    raise FileExistsError(
        f"Destination already exists:\n{FINAL_DIR}\n"
        "Rename it or choose another destination before running."
    )

files = sorted(glob.glob(os.path.join(SOURCE_DIR, "*.parquet")))
if not files:
    raise FileNotFoundError("No Parquet files found.")

# 2. Validate schemas before writing anything
source_columns = None
source_rows = 0

for file in files:
    pf = pq.ParquetFile(file)
    cols = pf.schema_arrow.names

    missing = [c for c in REMOVE if c not in cols]
    if missing:
        raise ValueError(
            f"{os.path.basename(file)} is missing: {missing}"
        )

    if source_columns is None:
        source_columns = cols
    elif cols != source_columns:
        raise ValueError(
            f"Schema mismatch: {os.path.basename(file)}"
        )

    source_rows += pf.metadata.num_rows

keep_columns = [c for c in source_columns if c not in REMOVE]
os.makedirs(FINAL_DIR)

try:
    # 3. Write the new dataset without the four columns
    for i, file in enumerate(files, 1):
        pf = pq.ParquetFile(file)
        output_file = os.path.join(
            FINAL_DIR, os.path.basename(file)
        )

        writer = None
        try:
            for batch in pf.iter_batches(
                batch_size=50_000,
                columns=keep_columns
            ):
                table = pa.Table.from_batches([batch])

                if writer is None:
                    writer = pq.ParquetWriter(
                        output_file,
                        table.schema,
                        compression="snappy"
                    )

                writer.write_table(table)

            if writer is None:
                empty_table = pa.table({
                    c: pa.array(
                        [], type=pf.schema_arrow.field(c).type
                    )
                    for c in keep_columns
                })
                pq.write_table(
                    empty_table, output_file, compression="snappy"
                )
        finally:
            if writer is not None:
                writer.close()

        if i % 10 == 0 or i == len(files):
            print(f"Written {i}/{len(files)} files")

    # 4. Verify output
    final_files = sorted(
        glob.glob(os.path.join(FINAL_DIR, "*.parquet"))
    )

    if len(final_files) != len(files):
        raise RuntimeError("File count mismatch.")

    final_rows = 0

    for file in final_files:
        pf = pq.ParquetFile(file)
        final_rows += pf.metadata.num_rows
        cols = pf.schema_arrow.names

        if any(c in cols for c in REMOVE):
            raise RuntimeError(
                f"Removed column still exists in {file}"
            )

        if cols != keep_columns:
            raise RuntimeError(
                f"Unexpected schema in {os.path.basename(file)}"
            )

    if final_rows != source_rows:
        raise RuntimeError("Row count mismatch.")

    print("\n===== FINAL VALIDATION PASSED =====")
    print("Files:", len(final_files))
    print("Rows:", f"{final_rows:,}")
    print("Columns before:", len(source_columns))
    print("Columns after:", len(keep_columns))
    print("Removed:", REMOVE)
    print("Output:", FINAL_DIR)

except Exception:
    shutil.rmtree(FINAL_DIR, ignore_errors=True)
    raise


KeyboardInterrupt: 